# Setup

In [ ]:
!pip uninstall numpy -y
!pip install "numpy<2.0" --force-reinstall

In [ ]:
!pip install \
    pillow \
    numpy \
    matplotlib \
    shapely \
    requests \
    transformers \
    accelerate \
    bitsandbytes \
    sentencepiece \
    safetensors

!pip install opencv-python-headless

In [ ]:
%%bash
git lfs install

In [ ]:
%%bash
git clone https://huggingface.co/datasets/xiang709/VRSBench

#kill this cell manually after getting the images_val.zip because for some reason
#huggingface takes FOREVER to get the train images and we don't need them anyways

In [ ]:
import zipfile
for z in ["VRSBench/Images_val.zip", "VRSBench/Annotations_val.zip"]:
    with zipfile.ZipFile(z, 'r') as zip_ref:
        zip_ref.extractall("VRSBench_val")

In [ ]:
download_link = "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth"
output_filename = "sam_vit_h_4b8939.pth"
!wget -O $output_filename "$download_link"

In [ ]:
!git clone https://huggingface.co/erenzhou/GeoGround

In [ ]:
!pip uninstall transformers -y
!pip install transformers==4.35.2 accelerate==0.24.1

In [ ]:
import os
import sys

# Clone and install LLaVA
!git clone https://github.com/haotian-liu/LLaVA.git

# Save current directory
original_dir = os.getcwd()

# Change to LLaVA directory
llava_dir = os.path.join(original_dir, "LLaVA")
os.chdir(llava_dir)

# Install
!pip install --upgrade pip
!pip install -e . --no-deps
!pip install protobuf bitsandbytes

# Return to original directory
os.chdir(original_dir)

# Add LLaVA to Python path
sys.path.insert(0, llava_dir)

# Models

In [ ]:
import os

# Use relative paths that work anywhere
BASE_DIR = os.getcwd()
IMAGE_FOLDER = os.path.join(BASE_DIR, "VRSBench_val/Images_val")
ANNOT_FOLDER = os.path.join(BASE_DIR, "VRSBench_val/Annotations_val")
SAM_CHECKPOINT = os.path.join(BASE_DIR, "sam_vit_h_4b8939.pth")
MODEL_PATH = os.path.join(BASE_DIR, "GeoGround/llava-v1.5-7b-task-lora-geoground")

In [ ]:
import os
import json
import torch
import requests
from PIL import Image
from io import BytesIO
from tqdm import tqdm
from transformers import AutoTokenizer, BitsAndBytesConfig
from llava.model import LlavaLlamaForCausalLM
from llava.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN
from llava.conversation import conv_templates
from llava.mm_utils import tokenizer_image_token, process_images

MODEL_PATH = "GeoGround/llava-v1.5-7b-task-lora-geoground"

# --- 1. LOAD MODEL IN 8-BIT ---
print("Loading model in 8-bit precision...")

# 8-bit Config (Int8)
quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=False)
model = LlavaLlamaForCausalLM.from_pretrained(
    MODEL_PATH,
    low_cpu_mem_usage=True,
    quantization_config=quantization_config,
    device_map="auto"
)

vision_tower = model.get_vision_tower()
if not vision_tower.is_loaded:
    vision_tower.load_model()
vision_tower.to(device='cuda', dtype=torch.float16)
image_processor = vision_tower.image_processor

print("SUCCESS: Model loaded!")

In [ ]:
try:
    import segment_anything
except ImportError:
    print("Installing segment-anything...")
    !pip install -q segment-anything
    import segment_anything

SAM_CHECKPOINT = "sam_vit_h_4b8939.pth"
SAM_MODEL_TYPE = "vit_h" # This corresponds to the 4b8939 weights

from segment_anything import sam_model_registry, SamPredictor

SAM_CHECKPOINT = "sam_vit_h_4b8939.pth"
SAM_MODEL_TYPE = "vit_h"

device = "cuda" if torch.cuda.is_available() else "cpu"
sam = sam_model_registry[SAM_MODEL_TYPE](checkpoint=SAM_CHECKPOINT)
sam.to(device=device)

# IMPORTANT: Make this global so evaluation loop can access it
global sam_predictor
sam_predictor = SamPredictor(sam)
print("SAM model loaded successfully!")

# Inference on VRS

In [ ]:
import os
import json
import torch
import cv2
import numpy as np
from PIL import Image
from tqdm import tqdm
from shapely.geometry import Polygon
import warnings
from datetime import datetime, timedelta
import gc
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Suppress warnings
warnings.filterwarnings('ignore', category=UserWarning, module='transformers.generation.configuration_utils')
warnings.filterwarnings('ignore', category=DeprecationWarning)

# Setup paths
BASE_DIR = os.getcwd()
IMAGE_FOLDER = os.path.join(BASE_DIR, "VRSBench_val/Images_val")
ANNOT_FOLDER = os.path.join(BASE_DIR, "VRSBench_val/Annotations_val")

# Verify directories exist
assert os.path.exists(IMAGE_FOLDER), f"Image folder not found: {IMAGE_FOLDER}"
assert os.path.exists(ANNOT_FOLDER), f"Annotation folder not found: {ANNOT_FOLDER}"

# Checkpoint configuration
CHECKPOINT_FILE = "evaluation_checkpoint.json"
CHECKPOINT_INTERVAL = 100  # Save every 100 images
VIZ_INTERVAL = 15  # Visualize every 15 images
BBOX_LOG_FILE = "bbox_predictions.json"  # Store bbox predictions
BBOX_LOG_INTERVAL = 50  # Save bbox data every 50 images

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def get_obb_from_mask(mask):
    """Extract OBB from binary mask - FIXED for deprecation warning."""
    if isinstance(mask, torch.Tensor):
        mask_np = mask.cpu().numpy().astype(np.uint8)
    else:
        mask_np = mask.astype(np.uint8)
    
    contours, _ = cv2.findContours(mask_np, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if len(contours) == 0:
        return None, None
    
    largest_contour = max(contours, key=cv2.contourArea)
    rect = cv2.minAreaRect(largest_contour)
    box_points = cv2.boxPoints(rect)
    box_points = np.intp(box_points)  # Fixed: use np.intp instead of np.int0
    return rect, box_points


def calculate_iou(pred_poly_pts, gt_poly_pts):
    """Calculate IoU between two polygons."""
    try:
        poly1 = Polygon(pred_poly_pts)
        poly2 = Polygon(gt_poly_pts)
        
        if not poly1.is_valid:
            poly1 = poly1.buffer(0)
        if not poly2.is_valid:
            poly2 = poly2.buffer(0)
        
        intersection_area = poly1.intersection(poly2).area
        union_area = poly1.area + poly2.area - intersection_area
        
        return intersection_area / union_area if union_area > 0 else 0.0
    except:
        return 0.0


def parse_bbox(text):
    """Extract normalized bbox coordinates from text."""
    import re
    numbers = re.findall(r"[-+]?\d*\.\d+|\d+", text)
    coords = [float(n) for n in numbers]
    
    if len(coords) >= 4:
        coords = coords[-4:]
        # Normalize if needed
        if any(c > 1.1 for c in coords):
            coords = [c / 1000.0 for c in coords]
        return coords
    return None


def get_sam_mask(image_pil, bbox_norm):
    """Get SAM segmentation mask from bbox."""
    image_np = np.array(image_pil)
    sam_predictor.set_image(image_np)
    
    H, W = image_np.shape[:2]
    box_pixels = np.array([
        bbox_norm[0] * W, bbox_norm[1] * H,
        bbox_norm[2] * W, bbox_norm[3] * H
    ])
    
    masks, _, _ = sam_predictor.predict(
        point_coords=None,
        point_labels=None,
        box=box_pixels[None, :],
        multimask_output=False
    )
    return masks[0]


def visualize_predictions(image_pil, gt_points, pred_points, iou, question, image_id):
    """Visualize ground truth and predicted OBBs on the image."""
    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    
    # Display image
    ax.imshow(image_pil)
    
    # Draw Ground Truth OBB (Green)
    if gt_points is not None and len(gt_points) > 0:
        gt_poly = patches.Polygon(
            gt_points, 
            linewidth=3, 
            edgecolor='green', 
            facecolor='none',
            label='Ground Truth'
        )
        ax.add_patch(gt_poly)
    
    # Draw Predicted OBB (Red)
    if pred_points is not None and len(pred_points) > 0:
        pred_poly = patches.Polygon(
            pred_points, 
            linewidth=3, 
            edgecolor='red', 
            facecolor='none',
            linestyle='--',
            label='Predicted'
        )
        ax.add_patch(pred_poly)
    
    # Add title and legend
    title = f"ID: {image_id}\nQuery: {question[:80]}...\nIoU: {iou:.4f}"
    if iou >= 0.7:
        title += " ✓ (≥0.7)"
    elif iou >= 0.5:
        title += " ~ (≥0.5)"
    else:
        title += " ✗ (<0.5)"
    
    ax.set_title(title, fontsize=10, pad=10)
    ax.legend(loc='upper right', fontsize=10)
    ax.axis('off')
    
    plt.tight_layout()
    plt.show()
    plt.close()


# ============================================================================
# CHECKPOINT LOADING
# ============================================================================

if os.path.exists(CHECKPOINT_FILE):
    print(f"📂 Found checkpoint file, loading...")
    with open(CHECKPOINT_FILE, 'r') as f:
        checkpoint = json.load(f)
        processed_ids = set(checkpoint.get('processed_ids', []))
        ious = checkpoint.get('ious', [])
        success_05 = checkpoint.get('success_05', 0)
        success_07 = checkpoint.get('success_07', 0)
        total_samples = checkpoint.get('total_samples', 0)
        failed_samples = checkpoint.get('failed_samples', [])
        viz_buffer = checkpoint.get('viz_buffer', [])  # For visualization tracking
    print(f"✅ Resuming from checkpoint: {total_samples} samples already processed")
else:
    print("🆕 Starting fresh evaluation")
    processed_ids = set()
    ious = []
    success_05 = 0
    success_07 = 0
    total_samples = 0
    failed_samples = []
    viz_buffer = []

# Load existing bbox predictions if available
if os.path.exists(BBOX_LOG_FILE):
    with open(BBOX_LOG_FILE, 'r') as f:
        bbox_predictions = json.load(f)
    print(f"📦 Loaded {len(bbox_predictions)} existing bbox predictions")
else:
    bbox_predictions = []

# ============================================================================
# MAIN EVALUATION LOOP
# ============================================================================

# Get all annotation files
json_files = sorted([f for f in os.listdir(ANNOT_FOLDER) if f.endswith('.json')])
print(f"📊 Total annotation files: {len(json_files)}")
print(f"⏭️  Already processed: {len(processed_ids)}")
print(f"🎯 Remaining: {len(json_files) - len(processed_ids)}\n")

start_time = datetime.now()

for json_file in tqdm(json_files, desc="Evaluating", initial=len(processed_ids), total=len(json_files)):
    image_id = json_file.replace('.json', '')
    
    # Skip if already processed
    if image_id in processed_ids:
        continue
    
    try:
        # Load annotation
        file_path = os.path.join(ANNOT_FOLDER, json_file)
        with open(file_path, 'r') as f:
            data = json.load(f)
        
        # Find image file
        image_path = None
        for ext in ['.png', '.jpg', '.jpeg', '.PNG', '.JPG', '.JPEG', '.bmp', '.tif']:
            temp_path = os.path.join(IMAGE_FOLDER, f"{image_id}{ext}")
            if os.path.exists(temp_path):
                image_path = temp_path
                break
        
        if not image_path:
            failed_samples.append((image_id, "Image file not found"))
            continue
        
        # Load image
        raw_image = Image.open(image_path).convert('RGB')
        width, height = raw_image.size
        image_tensor = process_images([raw_image], image_processor, model.config).to(
            model.device, dtype=torch.float16
        )
        
        # Get first object
        objects = data.get('objects', [])
        if not objects:
            failed_samples.append((image_id, "No objects in annotation"))
            continue
        
        obj = objects[0]
        question = obj['referring_sentence']
        
        # Prepare GT polygon (convert normalized coords to pixels)
        gt_raw = obj['obj_corner']
        gt_pixels = [(gt_raw[i] * width, gt_raw[i+1] * height) 
                     for i in range(0, len(gt_raw), 2)]
        
        # ========================================================================
        # GeoGround Inference (HBB prediction)
        # ========================================================================
        qs = f"[refer] output the bounding box of the <ref>{question}</ref> in the image."
        if model.config.mm_use_im_start_end:
            qs = DEFAULT_IM_START_TOKEN + DEFAULT_IMAGE_TOKEN + DEFAULT_IM_END_TOKEN + '\n' + qs
        else:
            qs = DEFAULT_IMAGE_TOKEN + '\n' + qs
        
        conv = conv_templates["llava_v1"].copy()
        conv.append_message(conv.roles[0], qs)
        conv.append_message(conv.roles[1], None)
        prompt = conv.get_prompt()
        
        input_ids = tokenizer_image_token(
            prompt, tokenizer, IMAGE_TOKEN_INDEX, return_tensors='pt'
        ).unsqueeze(0).cuda()
        
        with torch.inference_mode():
            output_ids = model.generate(
                input_ids, 
                images=image_tensor, 
                do_sample=False,
                max_new_tokens=256, 
                use_cache=True
            )
        
        output_text = tokenizer.batch_decode(output_ids, skip_special_tokens=True)[0].strip()
        pred_hbb_norm = parse_bbox(output_text)
        
        # ========================================================================
        # SAM Segmentation + OBB Extraction
        # ========================================================================
        iou = 0.0
        pred_obb_points = None
        
        if pred_hbb_norm:
            try:
                # Get segmentation mask from SAM
                sam_mask = get_sam_mask(raw_image, pred_hbb_norm)
                
                # Extract OBB from mask
                rect, pred_obb_points = get_obb_from_mask(sam_mask)
                
                if pred_obb_points is not None:
                    # Calculate IoU between predicted OBB and GT polygon
                    iou = calculate_iou(pred_obb_points, gt_pixels)
            except Exception as e:
                # SAM or OBB extraction failed, keep iou=0.0
                pass
        
        # ========================================================================
        # Store visualization data
        # ========================================================================
        viz_buffer.append({
            'image': raw_image.copy(),
            'gt_points': gt_pixels,
            'pred_points': pred_obb_points,
            'iou': iou,
            'question': question,
            'image_id': image_id
        })
        
        # ========================================================================
        # Store bbox prediction data
        # ========================================================================
        bbox_entry = {
            'image_id': image_id,
            'referring_sentence': question,
            'ground_truth_bbox': gt_pixels,  # List of (x, y) tuples
            'predicted_bbox': pred_obb_points.tolist() if pred_obb_points is not None else None,
            'iou': float(iou),
            'timestamp': datetime.now().isoformat()
        }
        bbox_predictions.append(bbox_entry)
        
        # ========================================================================
        # Update Metrics
        # ========================================================================
        ious.append(iou)
        total_samples += 1
        if iou >= 0.5:
            success_05 += 1
        if iou >= 0.7:
            success_07 += 1
        
        # Mark as processed
        processed_ids.add(image_id)
        
        # ========================================================================
        # VISUALIZATION (every VIZ_INTERVAL samples)
        # ========================================================================
        if len(viz_buffer) >= VIZ_INTERVAL:
            print(f"\n{'='*70}")
            print(f"📸 VISUALIZATION BATCH (Samples {total_samples - VIZ_INTERVAL + 1} to {total_samples})")
            print(f"{'='*70}")
            
            # Calculate Acc@0.7 for this batch
            batch_iou_07 = sum(1 for v in viz_buffer if v['iou'] >= 0.7)
            batch_acc_07 = batch_iou_07 / len(viz_buffer)
            
            # Overall Acc@0.7
            overall_acc_07 = success_07 / total_samples if total_samples > 0 else 0
            
            print(f"📊 Batch Acc@0.7: {batch_acc_07:.2%} ({batch_iou_07}/{len(viz_buffer)})")
            print(f"📊 Overall Acc@0.7: {overall_acc_07:.2%} ({success_07}/{total_samples})")
            print(f"{'='*70}\n")
            
            # Visualize all images in buffer
            for viz_data in viz_buffer:
                visualize_predictions(
                    viz_data['image'],
                    viz_data['gt_points'],
                    viz_data['pred_points'],
                    viz_data['iou'],
                    viz_data['question'],
                    viz_data['image_id']
                )
            
            # Clear buffer
            viz_buffer = []
        
        # ========================================================================
        # Progress Updates (every 10 samples)
        # ========================================================================
        if total_samples % 10 == 0 and total_samples > 0:
            elapsed = (datetime.now() - start_time).total_seconds()
            speed = elapsed / total_samples
            remaining_samples = len(json_files) - len(processed_ids)
            remaining_time = speed * remaining_samples
            eta = datetime.now() + timedelta(seconds=remaining_time)
            current_acc_05 = success_05 / total_samples
            current_acc_07 = success_07 / total_samples
            
            print(f"\n📊 Progress: {total_samples}/{len(json_files)} | "
                  f"Acc@0.5: {current_acc_05:.2%} | "
                  f"Acc@0.7: {current_acc_07:.2%} | "
                  f"Speed: {speed:.2f}s/img | "
                  f"ETA: {eta.strftime('%Y-%m-%d %H:%M')}")
        
        # ========================================================================
        # Save BBox Predictions (every BBOX_LOG_INTERVAL samples)
        # ========================================================================
        if total_samples % BBOX_LOG_INTERVAL == 0:
            with open(BBOX_LOG_FILE, 'w') as f:
                json.dump(bbox_predictions, f, indent=2)
            print(f"📦 BBox predictions saved: {len(bbox_predictions)} entries in {BBOX_LOG_FILE}")
        
        # ========================================================================
        # Save Checkpoint (every N samples)
        # ========================================================================
        if total_samples % CHECKPOINT_INTERVAL == 0:
            checkpoint_data = {
                'processed_ids': list(processed_ids),
                'ious': ious,
                'success_05': success_05,
                'success_07': success_07,
                'total_samples': total_samples,
                'failed_samples': failed_samples,
                'viz_buffer': []  # Don't save viz buffer in checkpoint
            }
            with open(CHECKPOINT_FILE, 'w') as f:
                json.dump(checkpoint_data, f)
            print(f"💾 Checkpoint saved at {total_samples} samples")
        
        # ========================================================================
        # Memory Management (every 50 samples)
        # ========================================================================
        if total_samples % 50 == 0 and torch.cuda.is_available():
            torch.cuda.empty_cache()
            gc.collect()
    
    except Exception as e:
        failed_samples.append((image_id, f"{type(e).__name__}: {str(e)}"))
        continue

# ============================================================================
# VISUALIZE REMAINING BUFFER (if any)
# ============================================================================
if viz_buffer:
    print(f"\n{'='*70}")
    print(f"📸 FINAL VISUALIZATION BATCH (Last {len(viz_buffer)} samples)")
    print(f"{'='*70}")
    
    batch_iou_07 = sum(1 for v in viz_buffer if v['iou'] >= 0.7)
    batch_acc_07 = batch_iou_07 / len(viz_buffer)
    overall_acc_07 = success_07 / total_samples if total_samples > 0 else 0
    
    print(f"📊 Batch Acc@0.7: {batch_acc_07:.2%} ({batch_iou_07}/{len(viz_buffer)})")
    print(f"📊 Overall Acc@0.7: {overall_acc_07:.2%} ({success_07}/{total_samples})")
    print(f"{'='*70}\n")
    
    for viz_data in viz_buffer:
        visualize_predictions(
            viz_data['image'],
            viz_data['gt_points'],
            viz_data['pred_points'],
            viz_data['iou'],
            viz_data['question'],
            viz_data['image_id']
        )

# ============================================================================
# FINAL CHECKPOINT SAVE
# ============================================================================
checkpoint_data = {
    'processed_ids': list(processed_ids),
    'ious': ious,
    'success_05': success_05,
    'success_07': success_07,
    'total_samples': total_samples,
    'failed_samples': failed_samples,
    'completed': True
}
with open(CHECKPOINT_FILE, 'w') as f:
    json.dump(checkpoint_data, f)

# Save final bbox predictions
with open(BBOX_LOG_FILE, 'w') as f:
    json.dump(bbox_predictions, f, indent=2)
print(f"\n📦 Final bbox predictions saved: {len(bbox_predictions)} entries")

# ============================================================================
# FINAL RESULTS
# ============================================================================
print("\n" + "="*60)
print("🎉 EVALUATION COMPLETE!")
print("="*60)
print(f"✅ Successfully Processed: {total_samples}")
print(f"❌ Failed Samples: {len(failed_samples)}")
print(f"📈 Mean IoU: {np.mean(ious):.4f}")
print(f"🎯 Acc@0.5 (IoU >= 0.5): {(success_05/total_samples):.2%}" if total_samples > 0 else "N/A")
print(f"🎯 Acc@0.7 (IoU >= 0.7): {(success_07/total_samples):.2%}" if total_samples > 0 else "N/A")

total_time = datetime.now() - start_time
print(f"⏱️  Total Time: {str(total_time).split('.')[0]}")
print(f"⚡ Average Speed: {total_time.total_seconds()/total_samples:.2f}s per image" if total_samples > 0 else "N/A")
print("="*60)

if failed_samples and len(failed_samples) <= 20:
    print("\n⚠️  Failed Samples:")
    for img_id, reason in failed_samples:
        print(f"  • {img_id}: {reason}")
elif failed_samples:
    print(f"\n⚠️  First 20 Failed Samples (out of {len(failed_samples)}):")
    for img_id, reason in failed_samples[:20]:
        print(f"  • {img_id}: {reason}")

print(f"\n💾 Full results saved to: {CHECKPOINT_FILE}")
print(f"📦 BBox predictions saved to: {BBOX_LOG_FILE}")
print(f"\n📄 BBox JSON format:")
print(f"  - image_id: Image identifier")
print(f"  - referring_sentence: The query text")
print(f"  - ground_truth_bbox: List of [x, y] coordinates for GT OBB corners")
print(f"  - predicted_bbox: List of [x, y] coordinates for predicted OBB corners")
print(f"  - iou: IoU score (0.0 to 1.0)")
print(f"  - timestamp: When the prediction was made")